In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [22]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [23]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [35]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
batch_size = 1000
# batch_size = None

opt = torch_numopt.NewtonRaphson(
    params=model.parameters(),
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    daming=True,
    mu=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    batch_size=batch_size
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.06211883947253227
epoch:  1, loss: 0.05666238069534302
epoch:  2, loss: 0.04934896528720856
epoch:  3, loss: 0.04266594350337982
epoch:  4, loss: 0.038637928664684296
epoch:  5, loss: 0.034378986805677414
epoch:  6, loss: 0.03092673234641552
epoch:  7, loss: 0.014811394736170769
epoch:  8, loss: 0.010730323381721973
epoch:  9, loss: 0.008443387225270271
epoch:  10, loss: 0.008064846508204937
epoch:  11, loss: 0.005953130777925253
epoch:  12, loss: 0.005201864056289196
epoch:  13, loss: 0.0047079226933419704
epoch:  14, loss: 0.004319547209888697
epoch:  15, loss: 0.0040529146790504456
epoch:  16, loss: 0.0038004720117896795
epoch:  17, loss: 0.0035774242132902145
epoch:  18, loss: 0.00339876813814044
epoch:  19, loss: 0.0032443569507449865
epoch:  20, loss: 0.003111872123554349
epoch:  21, loss: 0.0030012892093509436
epoch:  22, loss: 0.002912406576797366
epoch:  23, loss: 0.002833128906786442
epoch:  24, loss: 0.0027437179815024137
epoch:  25, loss: 0.00266373855993

In [33]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7627272545229765
Test metrics:  R2 = 0.5612882956242324
